## Module 1: Graph Structure for Accessible Hubs, Cycle Detection and Shortest Route

In [1]:
import csv

class DSALinkedListNode: # class for nodes
    def __init__(self, value):
        self.value = value
        self.next = None

class DSALinkedList:  # class for single linked list structure
    def __init__(self):
        self.head = None

    def insertLast(self, value):
        newNode = DSALinkedListNode(value)
        if self.head is None:
            self.head = newNode
        
        else:
            currentNode = self.head
            while currentNode.next is not None:
                currentNode = currentNode.next
            currentNode.next = newNode

    def getLength(self): # to get the total number of nodes stored in the linked list which will help in shortest path
        count = 0
        current = self.head
        while current is not None:
            count = count + 1
            current = current.next
        return count

    def exist(self, label):   # to check the existence of labels and to avoid the duplicates
        current = self.head
        while current is not None:
            if current.value.label == label:   # the current.value.label here links to the vertex class of the graph
                return True
            current = current.next
        return False
    
    def getNode(self,label):  # to return the node
        current = self.head
        while current is not None:
            if current.value.label == label:
                return current.value
            current = current.next
        return None
        
        
class QueueNode:        # represent a single node in the queue. The job of indiviual queue node is where is the next node
        def __init__(self, value):
            self.value = value
            self.next = None        

class DSAQueue:  # overall queue manager that tracks and front and rear of the queue
        def __init__(self):
            self.front = None
            self.rear = None

        def enqueue(self,value):
            newNode = QueueNode(value)
            if self.rear is None:
                 self.front = newNode
                 self.rear = newNode
            else:
                 self.rear.next = newNode
                 self.rear = newNode

        def dequeue(self):
            if self.front is None:
                raise Exception("Queue underflow")
            value = self.front.value
            self.front = self.front.next
            if self.front is None:
                self.rear = None
            return value
        
        def isEmpty(self):
             return self.front is None

class StackNode:
    def __init__(self, value):
        self.value = value
        self.next = None

class DSAStack:
    def __init__(self):
        self.top = None

    def push(self, value):
        newNode = StackNode(value)
        newNode.next = self.top
        self.top = newNode

    def pop(self):
        if self.top is None:
            raise Exception("Stack underflow")
        value = self.top.value
        self.top = self.top.next
        return value

    def isEmpty(self):
        return self.top is None

    def toLinkedList(self):
        linkedList = DSALinkedList()
        current = self.top
        while current is not None:
            newNode = DSALinkedListNode(current.value)   # Insert at front to reverse order (stack top becomes tail)
            newNode.next = linkedList.head
            linkedList.head = newNode
            current = current.next
        return linkedList


# graph class structure

class Vertex: # this is vertex class  
        def __init__(self, label):
            self.label = label
            self.adjList = None

class EdgeNode:
     def __init__(self, dest, weight):
          self.dest = dest
          self.weight = weight
          self.next = None


class DistanceEntry:   # for shortest distance
     def __init__(self, label, distance):
          self.label = label
          self.distance = distance
          self.previous = None         # storing the location of the previous explored node if the next node is explored for backtracking

class BFSNode:
    def __init__(self, label, level, pathList):
        self.label = label        # current node label
        self.level = level        # current hop level
        self.pathList = pathList  # path to reach this label

class EdgeSortNode:
    def __init__(self, weight, dest):
        self.weight = weight
        self.dest = dest
        self.next = None

class EdgeSortList:
    def __init__(self):
        self.head = None

    def insertSorted(self, weight, dest):
        newNode = EdgeSortNode(weight, dest)
        if self.head is None or weight < self.head.weight:
            newNode.next = self.head
            self.head = newNode
        else:
            current = self.head
            while current.next is not None and current.next.weight <= weight:
                current = current.next
            newNode.next = current.next
            current.next = newNode  

class Graph:
    def __init__(self):
         self.vertices = DSALinkedList()

    def addVertex(self, label):
         if not self.vertices.exist(label):   # to check whether the vertex exist in existing DSAlinled list (self.vertices)
              self.vertices.insertLast(Vertex(label)) # if given vertex is not exiting, adding the vertex

    def addEdge(self, src, dest, weight):
        if not self.vertices.exist(src):
             self.addVertex(src)
        if not self.vertices.exist(dest):
             self.addVertex(dest)

        srcVertex = self.vertices.getNode(src)
        destVertex = self.vertices.getNode(dest)

        edge1= EdgeNode(dest, weight)
        edge1.next = srcVertex.adjList
        srcVertex.adjList = edge1

        edge2 = EdgeNode(src, weight)
        edge2.next = destVertex.adjList
        destVertex.adjList = edge2

    def isVisited(self, visitedList, label):
        current = visitedList.head
        while current is not None:
            if current.value == label:
                return True
            current = current.next
        return False
    
    def bfsWithPaths(self, startLabel):
        visited = DSALinkedList()
        q = DSAQueue()

        pathList = DSALinkedList()
        pathList.insertLast(startLabel)
        q.enqueue(BFSNode(startLabel, 0, pathList))

        print("\nBFS Traversal by Hop Level with Paths:")
        print("Node\tLevel\tPath")

        while not q.isEmpty():
            node = q.dequeue()
            currentLabel = node.label
            level = node.level
            path = node.pathList
            if not self.isVisited(visited, currentLabel):
                print(f"{currentLabel}\t{level}\t", end="")
                self.printPathLinkedList(path)
                visited.insertLast(currentLabel)

                currentVertex = self.vertices.getNode(currentLabel)

            edgeList = EdgeSortList()    # Sort edges by weight using custom linked list
            edge = currentVertex.adjList
            while edge is not None:
                if not self.isVisited(visited, edge.dest):
                    edgeList.insertSorted(edge.weight, edge.dest)
                edge = edge.next

            
            currentEdgeNode = edgeList.head   # enqueue sorted neighbors
            while currentEdgeNode is not None:
                newPath = self.copyPath(path)
                newPath.insertLast(currentEdgeNode.dest)
                q.enqueue(BFSNode(currentEdgeNode.dest, level + 1, newPath))
                currentEdgeNode = currentEdgeNode.next

    def copyPath(self, path):
        newPath = DSALinkedList()
        current = path.head
        while current is not None:
            newPath.insertLast(current.value)
            current = current.next
        return newPath


    def dfs(self):
        visited = DSALinkedList()
        print("\nDFS Traversal & Cycle Detection:")
        current = self.vertices.head
        while current is not None:
            vertex = current.value
            if not self.isVisited(visited, vertex.label):
                pathStack = DSAStack()
                if self.dfsVisit(vertex.label, visited, None, pathStack):
                    return
            current = current.next
        print("No cycles found.")

    def dfsVisit(self, label, visited, parent, pathStack):
        visited.insertLast(label)
        pathStack.push(label)

        currentVertex = self.vertices.getNode(label)
        edge = currentVertex.adjList

        while edge is not None:
            if not self.isVisited(visited, edge.dest):
                if self.dfsVisit(edge.dest, visited, label, pathStack):
                    return True
            elif edge.dest != parent:
                print("Cycle detected in the graph:")
                self.printCycleFromStack(pathStack, edge.dest, label)
                return True
            edge = edge.next

        pathStack.pop()    # for backtracking
        return False
    

    def printCycleFromStack(self, pathStack, startLabel, endLabel):
       
        path = pathStack.toLinkedList()   # converting stack to list to extract sub path

        current = path.head
        started = False
        reachedEnd = False

        while current is not None and not reachedEnd:
            val = current.value
            if val == startLabel:
                started = True
            if started:
                print(val, end=" -> ")
            if val == endLabel and started:
                reachedEnd = True
            current = current.next

        if started:
            print(startLabel)

    def getMinUnvisited(self, distanceList, visitedList):
        minEntry = None
        current = distanceList.head
        while current is not None:
            entry = current.value
            if not self.isVisited(visitedList, entry.label):
                if minEntry is None or entry.distance < minEntry.distance:
                    minEntry = entry
            current = current.next
        return minEntry
    
    def printPathLinkedList(self, pathList):
        current = pathList.head
        while current:
            print(current.value, end=" -> " if current.next else "")
            current = current.next
        print()


    def getDistanceEntry(self, distanceList, label):
        current = distanceList.head
        while current is not None:
            if current.value.label == label:
                return current.value
            current = current.next
        return None
    
    def buildPathList(self, distanceList, targetLabel):
        path = DSALinkedList()
        steps = 0
        maxSteps = self.vertices.getLength()

        current = self.getDistanceEntry(distanceList, targetLabel)
        while current is not None and steps < maxSteps:
            newNode = DSALinkedListNode(current.label)
            newNode.next = path.head
            path.head = newNode

            if current.previous is None:
                current = None
            else:
                current = self.getDistanceEntry(distanceList, current.previous)
            steps += 1

        return path

    
    def shortestPath(self, source):
        distanceList = DSALinkedList()
        visited = DSALinkedList()

        current = self.vertices.head
        while current is not None:
            vertex = current.value
            entry = DistanceEntry(vertex.label, float('inf'))
            if vertex.label == source:
                entry.distance = 0
            distanceList.insertLast(entry)
            current = current.next

        print(f"\nOptimized Shortest Paths in existing Network from '{source}' are:\n")

        currentEntry = self.getMinUnvisited(distanceList, visited)
        while currentEntry is not None:
            if not self.isVisited(visited, currentEntry.label):
                visited.insertLast(currentEntry.label)

                vertex = self.vertices.getNode(currentEntry.label)
                edge = vertex.adjList

                while edge is not None:
                    if not self.isVisited(visited, edge.dest):
                        neighbor = self.getDistanceEntry(distanceList, edge.dest)
                        newDist = currentEntry.distance + edge.weight
                        if newDist < neighbor.distance:
                            neighbor.distance = newDist
                            neighbor.previous = currentEntry.label
                    edge = edge.next

            currentEntry = self.getMinUnvisited(distanceList, visited)

        current = distanceList.head
        while current is not None:
            entry = current.value
            dist = entry.distance
            if dist == float('inf'):
                print(f"{entry.label} - Total Distance Time to Reach in minutes: inf")
                print("Path to Follow: Unreachable")
            else:
                print(f"{entry.label} - Total Distance Time to Reach in minutes: {dist}")
                print("Path to Follow:", end=" ")
                pathList = self.buildPathList(distanceList, entry.label)
                self.printPathLinkedList(pathList)
            print()
            current = current.next

        return distanceList

    def displayGraph(self):
        print("Graph Structure:")
        current = self.vertices.head
        while current is not None:
            vertex = current.value
            print(f"{vertex.label} -> ", end="")
            edge = vertex.adjList
            while edge is not None:
                print(f"{edge.dest}({edge.weight} minutes) -> ", end="")
                edge = edge.next
            print("End")
            current = current.next 



### Tester Code for Module 3

In [2]:
module_1 = Graph()

module_1.addVertex("A")
module_1.addVertex("B")
module_1.addVertex("C")
module_1.addVertex("D")
module_1.addVertex("E")
module_1.addVertex("F")
module_1.addVertex("G")
module_1.addVertex("H")
module_1.addVertex("I")

module_1.addEdge("A", "B", 5)
module_1.addEdge("B", "C", 10)
module_1.addEdge("C", "D", 15)
module_1.addEdge("D", "E", 20)
module_1.addEdge("E", "F", 25)
module_1.addEdge("B", "F", 30)
module_1.addEdge("F", "G", 35)
module_1.addEdge("G","H",40)

module_1.displayGraph()

Graph Structure:
A -> B(5 minutes) -> End
B -> F(30 minutes) -> C(10 minutes) -> A(5 minutes) -> End
C -> D(15 minutes) -> B(10 minutes) -> End
D -> E(20 minutes) -> C(15 minutes) -> End
E -> F(25 minutes) -> D(20 minutes) -> End
F -> G(35 minutes) -> B(30 minutes) -> E(25 minutes) -> End
G -> H(40 minutes) -> F(35 minutes) -> End
H -> G(40 minutes) -> End
I -> End


In [3]:
# to show the level of hops and path for each hop. Let's use A as source hub

module_1.bfsWithPaths("A")


BFS Traversal by Hop Level with Paths:
Node	Level	Path
A	0	A
B	1	A -> B
C	2	A -> B -> C
F	2	A -> B -> F
D	3	A -> B -> C -> D
E	3	A -> B -> F -> E
G	3	A -> B -> F -> G
H	4	A -> B -> F -> G -> H


In [4]:
# to check for the cycle detection in our structure

module_1.dfs()


DFS Traversal & Cycle Detection:
Cycle detected in the graph:
B -> F -> E -> D -> C -> B


In [5]:
# now calculating the efficient path. Again source hub is selected as "A"

shortest_route=module_1.shortestPath("A")


Optimized Shortest Paths in existing Network from 'A' are:

A - Total Distance Time to Reach in minutes: 0
Path to Follow: A

B - Total Distance Time to Reach in minutes: 5
Path to Follow: A -> B

C - Total Distance Time to Reach in minutes: 15
Path to Follow: A -> B -> C

D - Total Distance Time to Reach in minutes: 30
Path to Follow: A -> B -> C -> D

E - Total Distance Time to Reach in minutes: 50
Path to Follow: A -> B -> C -> D -> E

F - Total Distance Time to Reach in minutes: 35
Path to Follow: A -> B -> F

G - Total Distance Time to Reach in minutes: 70
Path to Follow: A -> B -> F -> G

H - Total Distance Time to Reach in minutes: 110
Path to Follow: A -> B -> F -> G -> H

I - Total Distance Time to Reach in minutes: inf
Path to Follow: Unreachable



In [6]:
# same above example but with no disconnected node and no cycle

test_graph = Graph()

test_graph.addVertex("A")
test_graph.addVertex("B")
test_graph.addVertex("C")
test_graph.addVertex("D")
test_graph.addVertex("E")
test_graph.addVertex("F")
test_graph.addVertex("G")
test_graph.addVertex("H")

test_graph.addEdge("A", "B", 5)
test_graph.addEdge("B", "C", 10)
test_graph.addEdge("C", "D", 15)
test_graph.addEdge("D", "E", 20)
test_graph.addEdge("E", "F", 25)
test_graph.addEdge("F", "G", 35)
test_graph.addEdge("G","H",40)

test_graph.displayGraph()
test_graph.bfsWithPaths("A")
test_graph.dfs()
test_graph.shortestPath("A")

Graph Structure:
A -> B(5 minutes) -> End
B -> C(10 minutes) -> A(5 minutes) -> End
C -> D(15 minutes) -> B(10 minutes) -> End
D -> E(20 minutes) -> C(15 minutes) -> End
E -> F(25 minutes) -> D(20 minutes) -> End
F -> G(35 minutes) -> E(25 minutes) -> End
G -> H(40 minutes) -> F(35 minutes) -> End
H -> G(40 minutes) -> End

BFS Traversal by Hop Level with Paths:
Node	Level	Path
A	0	A
B	1	A -> B
C	2	A -> B -> C
D	3	A -> B -> C -> D
E	4	A -> B -> C -> D -> E
F	5	A -> B -> C -> D -> E -> F
G	6	A -> B -> C -> D -> E -> F -> G
H	7	A -> B -> C -> D -> E -> F -> G -> H

DFS Traversal & Cycle Detection:
No cycles found.

Optimized Shortest Paths in existing Network from 'A' are:

A - Total Distance Time to Reach in minutes: 0
Path to Follow: A

B - Total Distance Time to Reach in minutes: 5
Path to Follow: A -> B

C - Total Distance Time to Reach in minutes: 15
Path to Follow: A -> B -> C

D - Total Distance Time to Reach in minutes: 30
Path to Follow: A -> B -> C -> D

E - Total Distance Time

## Module 2 - Hash Tables for Customer Information

In [7]:
import numpy as np
import csv

class DSAHashEntry:
    def __init__(self, ID, name, address, priority_level, delivery_status):
        self.ID = ID
        self.value1 = name
        self.value2 = address
        self.value3 = priority_level
        self.value4 = delivery_status
        self.state = 1

class DSAHashTable:

    def __init__(self,size,load_factor=0.7):
        self.load_factor = load_factor       # in case if user gives its own load factor
        self.min_load_factor = 0.25          # minimum load factor
        self.size=self.nextPrime(size)     # to make sure that the actual internal size of the table is always going to be prime compared to the user given size value.
        self.hashArray= np.empty(self.size, dtype = object)     # the hash array based on actual size that will be storing keys and values
        self.count = 0
        self.minSize = self.size
        self.collisions = 0

    def hash_shift_add_xor(self,ID):  #  hash function which is shift add and xor
        hash_val = 0
        for char in str(ID):
            hash_val ^= ((hash_val << 5) + (hash_val << 2) + ord(char))
        return hash_val
    
    def hash1(self, ID):   # for index
        return self.hash_shift_add_xor(ID) % self.size
     
    def hash2(self,ID):     # for step
        return 1 + (self.hash_shift_add_xor(ID)%(self.size - 1))    # step never become 0
    
    def loadfactor(self):
        return self.count/self.size
    
    # to insert a record manually
    def put(self, ID, name, address, priority_level, delivery_status):

        index = self.hash1(ID)
        step = self.hash2(ID)
        print(f"Inserting {ID} ---> index: {index}, step: {step}")
        i = 0

        while True:
            probeIndex = (index + i * step) % self.size
            entry = self.hashArray[probeIndex]

            if entry is None or entry.state ==0:   # not occupied/empty slot
                self.hashArray[probeIndex] = DSAHashEntry(ID, name, address, priority_level, delivery_status)
                self.count +=1
                if self.loadfactor() > self.load_factor:
                    self.resize()
                return

            if entry.ID == ID and entry.state == 1:  # to check for duplicate key and also whether value is occupied
                print(f"Updating key '{ID}'")   # to show that duplicates are being handled
                entry.value1 = name     # this will update the existing value and also for handling of duplicate values (old value overwritten)
                entry.value2 = address
                entry.value3 = priority_level
                entry.value4 = delivery_status
                return  
            
            print(f"Collision Occured at index {probeIndex}, which was already occupied by key {entry.ID}. Doing Probing")

            self.collisions+=1
            
            i += 1
            if i>=self.size:     # to get out of loop 
                raise Exception ("Hash Table is Full")
            
    
    def get(self, ID):    # to get the record against customer ID
        index = self.hash1(ID)
        step = self.hash2(ID)

        i = 0
        while True:
            probeIndex = (index + i * step) % self.size
            entry = self.hashArray[probeIndex]
            if entry is None:
                return "ID Not Found"
            if entry.ID == ID and entry.state == 1:
                return f"Customer Name: {entry.value1}, Delivery Hub: {entry.value2}, Priority Level: {entry.value3}, Delivery Status: {entry.value4}"
            i += 1
            if i >= self.size:
                return None  
            
    def getCustomerCount(self):  # gives total number of customers
        return self.count
            
    def remove(self, ID):
        index = self.hash1(ID)
        step = self.hash2(ID)
        i = 0

        while True:
            probeIndex = (index + i * step) % self.size
            entry = self.hashArray[probeIndex]

            if entry is None:
                return "ID not Found"

            if entry.ID == ID and entry.state == 1:
                entry.state = 0 
                self.count -= 1
                if self.loadfactor() < self.min_load_factor and self.size > self.minSize:
                    newSize = self.size // 2
                    if newSize < self.minSize:
                        newSize = self.minSize
                    self.resize(newSize)
                return "Value has been removed"

            i += 1
            if i >= self.size:
                return False
            
    def hasKey(self, ID):   # check the existence of the key
        index = self.hash1(ID)
        step = self.hash2(ID)
        i = 0

        while True:
            probeIndex = (index + i * step) % self.size
            entry = self.hashArray[probeIndex]

            if entry is None:
                return ("ID Not Found")

            if entry.ID == ID and entry.state == 1:
                return ("ID Exist")

            i += 1
            if i >= self.size:
                return False 
            
    def updateStatus(self, ID, new_status):   # to update the status
        index = self.hash1(ID)
        step = self.hash2(ID)
        i = 0

        while True:
            probeIndex = (index + i * step) % self.size
            entry = self.hashArray[probeIndex]

            if entry is None:
                return "ID Not Found"

            if entry.ID == ID and entry.state == 1:
                old_status = entry.value4
                entry.value4 = new_status

                if new_status == "delivered" or new_status == "Delivered":
                    entry.value3 = 5
                elif new_status == "transit" or new_status == "Transit":
                    entry.value3 = 3
                elif new_status == "delayed" or new_status == "Delayed" :
                    entry.value3 = 1

                return f"Updated status for {ID}: {old_status} ---> {new_status}"
        
            i += 1
            if i >= self.size:
                return "ID Not Found"
        
    def resize(self, newSize=None):
        oldArray = self.hashArray
        oldSize = self.size
        if newSize is None:
            self.size = self.nextPrime(oldSize * 2)
        else:
            self.size = self.nextPrime(newSize)

        self.hashArray = np.empty(self.size, dtype=object)
        self.count = 0

        for entry in oldArray:
            if entry is not None and entry.state == 1:
                self.put(entry.ID, entry.value1, entry.value2, entry.value3, entry.value4)

    def nextPrime(self, startVal):
        if startVal % 2 == 0:           
            primeVal = startVal + 1      
        else:
            primeVal = startVal
        primeVal = primeVal - 2
        isPrime = False
        while not isPrime:
            primeVal = primeVal + 2
            ii = 3
            isPrime = True
            while ii * ii <= primeVal and isPrime:
                if primeVal % ii == 0:
                    isPrime = False
                else:
                    ii = ii + 2
        return primeVal
    
    def displayTable(self):
        print("\nDelivery Hash Table with Customer Information\n")
        for i in range(self.size):
            entry = self.hashArray[i]
            if entry is not None and entry.state == 1:
                print(f"Index {i}: Customer ID: {entry.ID}, Customer Name: {entry.value1}, Address: {entry.value2}, Priority Level: {entry.value3}, Delivery Status: {entry.value4}")
            else:
                print(f"Index {i}: Empty Record")
    
    def export(self):
        exportData = ''
        for entry in self.hashArray:
            if entry is not None and entry.state == 1:
                exportData += f'{entry.ID},{entry.value1},{entry.value2},{entry.value3},{entry.value4}\n'
        return exportData
    
    def loadFromCSV(self, filename, encoding='utf-8-sig'):
        try:
            with open(filename, mode='r') as file:
                reader = csv.reader(file)
                for row in reader:
                    if len(row)==5:
                        ID = row[0].strip()
                        name = row[1].strip()
                        address = row[2].strip()
                        priority_level = int(row[3].strip())
                        delivery_status = row[4].strip()
                        self.put(ID, name, address, priority_level, delivery_status)
            return "CSV data successfully loaded."
        
        except FileNotFoundError:
            return f"File '{filename}' not found."
        except Exception as e:
            return f"An error occurred: {str(e)}"
             
   

### Creating a mock Hash Table

In [8]:
print("Creating hash table")
table = DSAHashTable(5)

table.put("A1", "John Smith", "Zone A", 5, "Delivered")
table.put("B1", "Alice Green", "Zone B", 3, "In transit")
table.put("C1", "Bob Lee", "Zone C", 1, "Pending")
table.displayTable()

Creating hash table
Inserting A1 ---> index: 4, step: 1
Inserting B1 ---> index: 3, step: 4
Inserting C1 ---> index: 1, step: 3

Delivery Hash Table with Customer Information

Index 0: Empty Record
Index 1: Customer ID: C1, Customer Name: Bob Lee, Address: Zone C, Priority Level: 1, Delivery Status: Pending
Index 2: Empty Record
Index 3: Customer ID: B1, Customer Name: Alice Green, Address: Zone B, Priority Level: 3, Delivery Status: In transit
Index 4: Customer ID: A1, Customer Name: John Smith, Address: Zone A, Priority Level: 5, Delivery Status: Delivered


In [9]:
# inserting a new customer

table.put("D1", "Waqas", "Zone D", 1, "Pending")
table.displayTable()

Inserting D1 ---> index: 2, step: 2
Inserting C1 ---> index: 7, step: 7
Inserting D1 ---> index: 6, step: 8
Inserting B1 ---> index: 9, step: 4
Inserting A1 ---> index: 3, step: 5

Delivery Hash Table with Customer Information

Index 0: Empty Record
Index 1: Empty Record
Index 2: Empty Record
Index 3: Customer ID: A1, Customer Name: John Smith, Address: Zone A, Priority Level: 5, Delivery Status: Delivered
Index 4: Empty Record
Index 5: Empty Record
Index 6: Customer ID: D1, Customer Name: Waqas, Address: Zone D, Priority Level: 1, Delivery Status: Pending
Index 7: Customer ID: C1, Customer Name: Bob Lee, Address: Zone C, Priority Level: 1, Delivery Status: Pending
Index 8: Empty Record
Index 9: Customer ID: B1, Customer Name: Alice Green, Address: Zone B, Priority Level: 3, Delivery Status: In transit
Index 10: Empty Record


In [10]:
# searching for a customer

table.get("C1")

'Customer Name: Bob Lee, Delivery Hub: Zone C, Priority Level: 1, Delivery Status: Pending'

In [11]:
# deleting a customer
table.remove("D1")
table.displayTable()


Delivery Hash Table with Customer Information

Index 0: Empty Record
Index 1: Empty Record
Index 2: Empty Record
Index 3: Customer ID: A1, Customer Name: John Smith, Address: Zone A, Priority Level: 5, Delivery Status: Delivered
Index 4: Empty Record
Index 5: Empty Record
Index 6: Empty Record
Index 7: Customer ID: C1, Customer Name: Bob Lee, Address: Zone C, Priority Level: 1, Delivery Status: Pending
Index 8: Empty Record
Index 9: Customer ID: B1, Customer Name: Alice Green, Address: Zone B, Priority Level: 3, Delivery Status: In transit
Index 10: Empty Record


In [12]:
# deleting another customer to show dynamic resizing in downsize
table.remove("C1")
table.displayTable()

Inserting A1 ---> index: 4, step: 1
Inserting B1 ---> index: 3, step: 4

Delivery Hash Table with Customer Information

Index 0: Empty Record
Index 1: Empty Record
Index 2: Empty Record
Index 3: Customer ID: B1, Customer Name: Alice Green, Address: Zone B, Priority Level: 3, Delivery Status: In transit
Index 4: Customer ID: A1, Customer Name: John Smith, Address: Zone A, Priority Level: 5, Delivery Status: Delivered


In [13]:
# showing collisions
table.put('Z9',"Kiyani", "Zone D", "1", "Delayed")
print(f"\nTotal no. of collisions are: {table.collisions}")
table.displayTable()


Inserting Z9 ---> index: 4, step: 4
Collision Occured at index 4, which was already occupied by key A1. Doing Probing
Collision Occured at index 3, which was already occupied by key B1. Doing Probing

Total no. of collisions are: 2

Delivery Hash Table with Customer Information

Index 0: Empty Record
Index 1: Empty Record
Index 2: Customer ID: Z9, Customer Name: Kiyani, Address: Zone D, Priority Level: 1, Delivery Status: Delayed
Index 3: Customer ID: B1, Customer Name: Alice Green, Address: Zone B, Priority Level: 3, Delivery Status: In transit
Index 4: Customer ID: A1, Customer Name: John Smith, Address: Zone A, Priority Level: 5, Delivery Status: Delivered


In [14]:
# added functionality to get total customers count
print(f"Total Customers are {table.getCustomerCount()}")


Total Customers are 3


In [15]:
# to update the status of a specific ID
table.get('A1')

'Customer Name: John Smith, Delivery Hub: Zone A, Priority Level: 5, Delivery Status: Delivered'

In [16]:
table.updateStatus('A1','Delayed')

'Updated status for A1: Delivered ---> Delayed'

In [17]:
# check the new status and also the priority level will auto adjust based on the new delivery status
table.get('A1')

'Customer Name: John Smith, Delivery Hub: Zone A, Priority Level: 1, Delivery Status: Delayed'

### Test: Showing Performance for more than 50 customers

In [18]:
module_2=DSAHashTable(5)
module_2.loadFromCSV('customers_record.csv')
module_2.displayTable()

Inserting 9358 ---> index: 1, step: 4
Inserting 4630 ---> index: 4, step: 2
Inserting 8451 ---> index: 0, step: 1
Inserting 6284 ---> index: 4, step: 1
Collision Occured at index 4, which was already occupied by key 4630. Doing Probing
Collision Occured at index 0, which was already occupied by key 8451. Doing Probing
Collision Occured at index 1, which was already occupied by key 9358. Doing Probing
Inserting 8451 ---> index: 6, step: 1
Inserting 9358 ---> index: 7, step: 2
Inserting 6284 ---> index: 0, step: 5
Inserting 4630 ---> index: 3, step: 10
Inserting 4631 ---> index: 2, step: 9
Inserting 6723 ---> index: 3, step: 3
Collision Occured at index 3, which was already occupied by key 4630. Doing Probing
Collision Occured at index 6, which was already occupied by key 8451. Doing Probing
Inserting 9413 ---> index: 7, step: 10
Collision Occured at index 7, which was already occupied by key 9358. Doing Probing
Collision Occured at index 6, which was already occupied by key 8451. Doing 

In [19]:
module_2.getCustomerCount()

75

In [20]:
print("\nExporting hash table to file...")
with open("exported_customers.csv", "w") as f:
    f.write(module_2.export())
    print("Export complete. File saved as 'exported_customers.csv'")


Exporting hash table to file...
Export complete. File saved as 'exported_customers.csv'


In [21]:
module_2.get('5863')

'Customer Name: SiennaRogers, Delivery Hub: B, Priority Level: 5, Delivery Status: Delayed'

In [22]:
module_2.updateStatus('5863','Delivered')

'Updated status for 5863: Delayed ---> Delivered'

In [23]:
module_2.get('5863')

'Customer Name: SiennaRogers, Delivery Hub: B, Priority Level: 5, Delivery Status: Delivered'

In [24]:
module_2.remove('5863')


'Value has been removed'

In [25]:
module_2.get('5863')

'ID Not Found'

## Module 3: Heap Structure for the Calculation of Priority

In [26]:
import numpy as np

class HeapNode:
    def __init__(self, customerID, priorityLevel, deliveryTime, deliveryStatus):
        self.customerID = customerID
        self.priorityLevel = priorityLevel
        self.deliveryTime = deliveryTime
        self.deliveryStatus = deliveryStatus
        self.updateScore()

    def update(self, priorityLevel, deliveryTime, deliveryStatus):
        self.priorityLevel = priorityLevel
        self.deliveryTime = deliveryTime
        self.deliveryStatus = deliveryStatus
        self.updateScore()

    def updateScore(self):
        if self.deliveryTime == 0 or self.deliveryStatus == 'delivered' or self.deliveryStatus == 'Delivered':
            self.priorityScore = 0
        else:
            self.priorityScore = (6 - self.priorityLevel) + (1000 / self.deliveryTime)

    def __str__(self):
        return f"ID: {self.customerID}, Score: {self.priorityScore:.2f}, Time: {self.deliveryTime}, Status: {self.deliveryStatus}"

class ParcelHeap:
    def __init__(self, capacity):
        self.heap = np.empty(capacity, dtype=object)
        self.size = 0
        self.capacity = capacity

    def insert(self, node):
        if self.size >= self.capacity:
            self.resizeHeap()
        self.heap[self.size] = node
        self.heapifyUp(self.size)
        self.size += 1

    def resizeHeap(self):
        newCapacity = self.capacity * 2
        newHeap = np.empty(newCapacity, dtype=object)
        for i in range(self.size):
            newHeap[i] = self.heap[i]
        self.heap = newHeap
        self.capacity = newCapacity

    def findRecordAndIndex(self, customerID):
        for i in range(self.size):
            if self.heap[i] and self.heap[i].customerID == customerID:
                return self.heap[i], i
        return None, -1

    def insertRequest(self, customerID, priorityLevel, deliveryTime, deliveryStatus):
        existing, idx = self.findRecordAndIndex(customerID)
        if existing:
            oldScore = existing.priorityScore
            existing.update(priorityLevel, deliveryTime, deliveryStatus)
            newScore = existing.priorityScore
            if newScore > oldScore:
                self.heapifyUp(idx)
            elif newScore < oldScore:
                self.heapifyDown(idx)
        else:
            parcel = HeapNode(customerID, priorityLevel, deliveryTime, deliveryStatus)
            self.insert(parcel)

    def extractMax(self):
        if self.size == 0:
            raise Exception("Heap is empty")
        maxNode = self.heap[0]
        self.size -= 1
        self.heap[0] = self.heap[self.size]
        self.heap[self.size] = None
        self.heapifyDown(0)
        return maxNode

    def heapifyUp(self, index):
        while index > 0 and self.heap[index].priorityScore > self.heap[(index - 1) // 2].priorityScore:
            parentIndex = (index - 1) // 2
            self.heap[index], self.heap[parentIndex] = self.heap[parentIndex], self.heap[index]
            index = parentIndex

    def heapifyDown(self, index):
        largest = index
        left = 2 * index + 1
        right = 2 * index + 2
        if left < self.size and self.heap[left].priorityScore > self.heap[largest].priorityScore:
            largest = left
        if right < self.size and self.heap[right].priorityScore > self.heap[largest].priorityScore:
            largest = right
        if largest != index:
            self.heap[index], self.heap[largest] = self.heap[largest], self.heap[index]
            self.heapifyDown(largest)

    def displayHeap(self):
        print("\nCurrent Heap State:")
        for i in range(self.size):
            print(f"Index {i}: {self.heap[i]}")


class Module3Test:
    def __init__(self, customerHashTable):
        self.customerHashTable = customerHashTable
        self.heap = ParcelHeap(10)

    def runTest(self, graph, hub):
        distanceList = graph.shortestPath(hub)

        print ('\nComplete Delivery Log with Customer Details is\n')
        for entry in self.customerHashTable.hashArray:
            if entry is not None and entry.state == 1:
                customerID = entry.ID
                customername = entry.value1
                destination = entry.value2
                priorityLevel = entry.value3
                deliveryStatus = entry.value4

                deliveryEntry = graph.getDistanceEntry(distanceList, destination)
                deliveryTime = deliveryEntry.distance if deliveryEntry is not None else float('inf')

                if deliveryTime == 0 or deliveryStatus == 'delivered' or deliveryStatus == 'Delivered':
                    calculatedScore = 0
                else:
                    calculatedScore = (6 - priorityLevel) + (1000 / deliveryTime)

                print(f"Parcel {customerID} | Customer Name: {customername:<15} | Zone: {destination} | "
                      f"Priority: {priorityLevel} | Time: {deliveryTime:<3} | "
                      f"Status: {deliveryStatus:<10} | Score: {calculatedScore:.2f}")

                self.heap.insertRequest(customerID, priorityLevel, deliveryTime, deliveryStatus)

        if self.heap.size == 0:
            print("Heap is empty — no customers to schedule.")
            return None, 0
        return self.heap.heap[0].customerID, self.heap.heap[0].priorityScore


In [27]:
# a mock test heap for independent testing

test_heap= ParcelHeap(3)
test_heap.insertRequest("C1", 2, 20, "In transit")     
test_heap.insertRequest("C2", 4, 10, "Transit")        
test_heap.insertRequest("C3", 3, 25, "Pending")        
test_heap.insertRequest("C4", 1, 5, "Transit")         
test_heap.insertRequest("C5", 5, 30, "In transit")     
test_heap.insertRequest("C6", 2, 0, "Delivered")       

In [28]:
test_heap.displayHeap()


Current Heap State:
Index 0: ID: C4, Score: 205.00, Time: 5, Status: Transit
Index 1: ID: C2, Score: 102.00, Time: 10, Status: Transit
Index 2: ID: C3, Score: 43.00, Time: 25, Status: Pending
Index 3: ID: C1, Score: 54.00, Time: 20, Status: In transit
Index 4: ID: C5, Score: 34.33, Time: 30, Status: In transit
Index 5: ID: C6, Score: 0.00, Time: 0, Status: Delivered


In [29]:
# for inserting a new request and heap state after new request
test_heap.insertRequest('C7',1,80,'Delayed')
test_heap.displayHeap()


Current Heap State:
Index 0: ID: C4, Score: 205.00, Time: 5, Status: Transit
Index 1: ID: C2, Score: 102.00, Time: 10, Status: Transit
Index 2: ID: C3, Score: 43.00, Time: 25, Status: Pending
Index 3: ID: C1, Score: 54.00, Time: 20, Status: In transit
Index 4: ID: C5, Score: 34.33, Time: 30, Status: In transit
Index 5: ID: C6, Score: 0.00, Time: 0, Status: Delivered
Index 6: ID: C7, Score: 17.50, Time: 80, Status: Delayed


In [30]:
print(test_heap.extractMax())

ID: C4, Score: 205.00, Time: 5, Status: Transit


In [31]:
test_heap.displayHeap()


Current Heap State:
Index 0: ID: C2, Score: 102.00, Time: 10, Status: Transit
Index 1: ID: C1, Score: 54.00, Time: 20, Status: In transit
Index 2: ID: C3, Score: 43.00, Time: 25, Status: Pending
Index 3: ID: C7, Score: 17.50, Time: 80, Status: Delayed
Index 4: ID: C5, Score: 34.33, Time: 30, Status: In transit
Index 5: ID: C6, Score: 0.00, Time: 0, Status: Delivered


In [32]:
# some real world functionality where a delivered parcel must have low priority score.
# Lets try for C2 by changing its delivery status only

test_heap.insertRequest('C2',1,10,'Delivered')
test_heap.displayHeap()


Current Heap State:
Index 0: ID: C1, Score: 54.00, Time: 20, Status: In transit
Index 1: ID: C5, Score: 34.33, Time: 30, Status: In transit
Index 2: ID: C3, Score: 43.00, Time: 25, Status: Pending
Index 3: ID: C7, Score: 17.50, Time: 80, Status: Delayed
Index 4: ID: C2, Score: 0.00, Time: 10, Status: Delivered
Index 5: ID: C6, Score: 0.00, Time: 0, Status: Delivered


In [33]:
print("\nRunning Parcel Heap Module")
module3 = Module3Test(module_2)
topCustomer, topScore= module3.runTest(module_1, "A")


Running Parcel Heap Module

Optimized Shortest Paths in existing Network from 'A' are:

A - Total Distance Time to Reach in minutes: 0
Path to Follow: A

B - Total Distance Time to Reach in minutes: 5
Path to Follow: A -> B

C - Total Distance Time to Reach in minutes: 15
Path to Follow: A -> B -> C

D - Total Distance Time to Reach in minutes: 30
Path to Follow: A -> B -> C -> D

E - Total Distance Time to Reach in minutes: 50
Path to Follow: A -> B -> C -> D -> E

F - Total Distance Time to Reach in minutes: 35
Path to Follow: A -> B -> F

G - Total Distance Time to Reach in minutes: 70
Path to Follow: A -> B -> F -> G

H - Total Distance Time to Reach in minutes: 110
Path to Follow: A -> B -> F -> G -> H

I - Total Distance Time to Reach in minutes: inf
Path to Follow: Unreachable


Complete Delivery Log with Customer Details is

Parcel 2941 | Customer Name: IsaacPrice      | Zone: G | Priority: 1 | Time: 70  | Status: Delivered  | Score: 0.00
Parcel 9041 | Customer Name: LiamEvans

In [34]:
module3.heap.displayHeap()


Current Heap State:
Index 0: ID: 5394, Score: 201.00, Time: 5, Status: InTransit
Index 1: ID: 5048, Score: 70.67, Time: 15, Status: InTransit
Index 2: ID: 2387, Score: 69.67, Time: 15, Status: InTransit
Index 3: ID: 4526, Score: 70.67, Time: 15, Status: Delayed
Index 4: ID: 2984, Score: 69.67, Time: 15, Status: Delayed
Index 5: ID: 9041, Score: 37.33, Time: 30, Status: Delayed
Index 6: ID: 3129, Score: 34.33, Time: 30, Status: Delayed
Index 7: ID: 8725, Score: 68.67, Time: 15, Status: InTransit
Index 8: ID: 7254, Score: 37.33, Time: 30, Status: Delayed
Index 9: ID: 2643, Score: 37.33, Time: 30, Status: InTransit
Index 10: ID: 8451, Score: 37.33, Time: 30, Status: Delayed
Index 11: ID: 1937, Score: 33.57, Time: 35, Status: InTransit
Index 12: ID: 1398, Score: 35.33, Time: 30, Status: InTransit
Index 13: ID: 5928, Score: 31.57, Time: 35, Status: Delayed
Index 14: ID: 5678, Score: 25.00, Time: 50, Status: InTransit
Index 15: ID: 1897, Score: 34.33, Time: 30, Status: Delayed
Index 16: ID:

In [35]:
print(f"Top Customer is: {topCustomer}")
print(f"Top Customer Score is: {topScore}")

Top Customer is: 5394
Top Customer Score is: 201.0


In [36]:
record, index = module3.heap.findRecordAndIndex('5394')
print(record)

ID: 5394, Score: 201.00, Time: 5, Status: InTransit


In [37]:
xx=module3.heap.extractMax()
print(xx)

print()
yy=module3.heap.extractMax()
print(yy)

print()
module3.heap.insertRequest('123',1,10,'Pending')

ID: 5394, Score: 201.00, Time: 5, Status: InTransit

ID: 5048, Score: 70.67, Time: 15, Status: InTransit



In [38]:
record1=module3.heap.findRecordAndIndex('123')

In [39]:
print(record1[0])

ID: 123, Score: 105.00, Time: 10, Status: Pending


In [40]:
module3.heap.insertRequest('123',5,5,'Delivered')

In [41]:
record2=module3.heap.findRecordAndIndex('123')
print(record2[0])

ID: 123, Score: 0.00, Time: 5, Status: Delivered


In [42]:
zz=module3.heap.extractMax()
print(zz)

ID: 4526, Score: 70.67, Time: 15, Status: Delayed


In [43]:
module3.heap.displayHeap()


Current Heap State:
Index 0: ID: 2984, Score: 69.67, Time: 15, Status: Delayed
Index 1: ID: 8725, Score: 68.67, Time: 15, Status: InTransit
Index 2: ID: 2387, Score: 69.67, Time: 15, Status: InTransit
Index 3: ID: 7162, Score: 37.33, Time: 30, Status: InTransit
Index 4: ID: 2643, Score: 37.33, Time: 30, Status: InTransit
Index 5: ID: 9041, Score: 37.33, Time: 30, Status: Delayed
Index 6: ID: 3129, Score: 34.33, Time: 30, Status: Delayed
Index 7: ID: 1897, Score: 34.33, Time: 30, Status: Delayed
Index 8: ID: 7254, Score: 37.33, Time: 30, Status: Delayed
Index 9: ID: 6239, Score: 35.33, Time: 30, Status: InTransit
Index 10: ID: 8451, Score: 37.33, Time: 30, Status: Delayed
Index 11: ID: 1937, Score: 33.57, Time: 35, Status: InTransit
Index 12: ID: 1398, Score: 35.33, Time: 30, Status: InTransit
Index 13: ID: 5928, Score: 31.57, Time: 35, Status: Delayed
Index 14: ID: 5678, Score: 25.00, Time: 50, Status: InTransit
Index 15: ID: 6745, Score: 33.57, Time: 35, Status: InTransit
Index 16: I

# Module 4: Sorting Delivery Records (Using Quick and Merge Sort)

In [44]:

import numpy as np
import csv
import time


class DeliveryRecord:
    def __init__(self, deliveryTime, address, deliveryID):
        self.deliveryTime = deliveryTime
        self.address = address
        self.deliveryID = deliveryID

class Sorter:
    def __init__(self):
        self.comparison = 0
        self.swap = 0

    def getKey(self, record, keyAttr):
        if keyAttr == "deliveryTime":
            return record.deliveryTime
        elif keyAttr == "address":
            return record.address
        elif keyAttr == "deliveryID":
            return record.deliveryID
        else:
            raise Exception("Invalid sorting key")

    def quickSort(self, A, keyAttr="deliveryTime"):
        self.comparison = 0
        self.swap = 0
        start = time.perf_counter()
        self.quickSortRecurse(A, 0, len(A) - 1, keyAttr)
        end = time.perf_counter()
        self.printSummary("QuickSort", end - start, size=len(A))

    def quickSortRecurse(self, A, leftIdx, rightIdx, keyAttr):
        if rightIdx > leftIdx:
            pivotIdx = (leftIdx + rightIdx) // 2
            newPivotIdx = self.doPartition(A, leftIdx, rightIdx, pivotIdx, keyAttr)
            self.quickSortRecurse(A, leftIdx, newPivotIdx - 1, keyAttr)
            self.quickSortRecurse(A, newPivotIdx + 1, rightIdx, keyAttr)

    def doPartition(self, A, leftIdx, rightIdx, pivotIdx, keyAttr):
        pivotVal = self.getKey(A[pivotIdx], keyAttr)
        A[pivotIdx], A[rightIdx] = A[rightIdx], A[pivotIdx]
        self.swap += 1
        currIdx = leftIdx
        for ii in range(leftIdx, rightIdx):
            self.comparison += 1
            if self.getKey(A[ii], keyAttr) < pivotVal:
                A[ii], A[currIdx] = A[currIdx], A[ii]
                self.swap += 1
                currIdx += 1
        A[rightIdx], A[currIdx] = A[currIdx], A[rightIdx]
        self.swap += 1
        return currIdx

    def mergeSort(self, A, keyAttr="deliveryTime"):
        self.comparison = 0
        self.swap = 0
        start = time.perf_counter()
        self.mergeSortRecurse(A, 0, len(A) - 1, keyAttr)
        end = time.perf_counter()
        self.printSummary("MergeSort", end - start, size=len(A))

    def mergeSortRecurse(self, A, leftIdx, rightIdx, keyAttr):
        if leftIdx < rightIdx:
            midIdx = (leftIdx + rightIdx) // 2
            self.mergeSortRecurse(A, leftIdx, midIdx, keyAttr)
            self.mergeSortRecurse(A, midIdx + 1, rightIdx, keyAttr)
            self.merge(A, leftIdx, midIdx, rightIdx, keyAttr)

    def merge(self, A, leftIdx, midIdx, rightIdx, keyAttr):
        tempArr = np.empty(rightIdx - leftIdx + 1, dtype=object)
        ii = leftIdx
        jj = midIdx + 1
        kk = 0
        while ii <= midIdx and jj <= rightIdx:
            self.comparison += 1
            if self.getKey(A[ii], keyAttr) <= self.getKey(A[jj], keyAttr):
                tempArr[kk] = A[ii]
                ii += 1
            else:
                tempArr[kk] = A[jj]
                jj += 1
            self.swap += 1
            kk += 1
        for idx in range(ii, midIdx + 1):
            tempArr[kk] = A[idx]
            self.swap += 1
            kk += 1
        for idx in range(jj, rightIdx + 1):
            tempArr[kk] = A[idx]
            self.swap += 1
            kk += 1
        for idx in range(leftIdx, rightIdx + 1):
            A[idx] = tempArr[idx - leftIdx]


    def printSummary(self, method, execTime, size=None): # to display the results
        print(f"\n{method.upper()} Performance\n")
        if size is not None:
            print(f"Total Records  : {size}")
        print(f"Execution Time: {execTime:.6f} sec")
        print(f"Number of Comparisons: {self.comparison}")
        print(f"Number of Swaps: {self.swap}")
        print()

    
    def makeArray(self, filename):
        linkedList = DSALinkedList()

        with open(filename, 'r') as file:
            reader = csv.reader(file)
            for row in reader:
                deliveryTime = row[0]
                address = row[1]
                deliveryID = row[2]
                record = DeliveryRecord(deliveryTime, address, deliveryID)
                linkedList.insertLast(record)

        arr = np.empty(linkedList.getLength(), dtype=object) # converting linked list to numpy array
        current = linkedList.head
        index = 0
        while current is not None:
            arr[index] = current.value
            current = current.next
            index += 1

        return arr

    def makeCSV(self, filename, array): # exporting results to csv
        with open(filename, 'w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(['DeliveryTime', 'Address', 'DeliveryID'])
            for i in range(len(array)):
                rec = array[i]
                writer.writerow([rec.deliveryTime, rec.address, rec.deliveryID])

# method for Mock Test Demo for all attributes (doing both sortings at one)

def mockSortTest(records, attribute = "deliverytime"):
        sorter = Sorter()
        quick_records = np.copy(records)
        merge_records = np.copy(records)

        # for quick sort
        print("Before Quick Sorting:\n")
        for r in quick_records:
            print(r.deliveryTime, r.address, r.deliveryID)
        sorter.quickSort(quick_records, keyAttr= attribute)
        print()

        print("After Quick Sorting:\n")
        for r in quick_records:
            print(r.deliveryTime, r.address, r.deliveryID)
        print()

        # for merge sort
        print("Before Merge Sorting:\n")
        for r in merge_records:
            print(r.deliveryTime, r.address, r.deliveryID)
        print()

        sorter.mergeSort(merge_records, keyAttr= attribute)
        print("After Merge Sorting:\n")
        for r in merge_records:
            print(r.deliveryTime, r.address, r.deliveryID)
        print()


### Test 1: Mock Record with Sort by Delivery Time

In [45]:
records = np.empty(10, dtype=object)
records[0] = DeliveryRecord(1300, "Zone C", "D003")
records[1] = DeliveryRecord(950, "Zone A", "D001")
records[2] = DeliveryRecord(1200, "Zone B", "D005")
records[3] = DeliveryRecord(5000, "Zone A", "D004")
records[4] = DeliveryRecord(1450, "Zone A", "D004")
records[5] = DeliveryRecord(500, "Zone B", "D005")
records[6] = DeliveryRecord(1100, "Zone A", "D001")
records[7] = DeliveryRecord(1300, "Zone C", "D003")
records[8] = DeliveryRecord(2000, "Zone D", "D002")
records[9] = DeliveryRecord(1000, "Zone D", "D002")

mockSortTest(records, attribute="deliveryTime")



Before Quick Sorting:

1300 Zone C D003
950 Zone A D001
1200 Zone B D005
5000 Zone A D004
1450 Zone A D004
500 Zone B D005
1100 Zone A D001
1300 Zone C D003
2000 Zone D D002
1000 Zone D D002

QUICKSORT Performance

Total Records  : 10
Execution Time: 0.000011 sec
Number of Comparisons: 21
Number of Swaps: 25


After Quick Sorting:

500 Zone B D005
950 Zone A D001
1000 Zone D D002
1100 Zone A D001
1200 Zone B D005
1300 Zone C D003
1300 Zone C D003
1450 Zone A D004
2000 Zone D D002
5000 Zone A D004

Before Merge Sorting:

1300 Zone C D003
950 Zone A D001
1200 Zone B D005
5000 Zone A D004
1450 Zone A D004
500 Zone B D005
1100 Zone A D001
1300 Zone C D003
2000 Zone D D002
1000 Zone D D002


MERGESORT Performance

Total Records  : 10
Execution Time: 0.000019 sec
Number of Comparisons: 24
Number of Swaps: 34

After Merge Sorting:

500 Zone B D005
950 Zone A D001
1000 Zone D D002
1100 Zone A D001
1200 Zone B D005
1300 Zone C D003
1300 Zone C D003
1450 Zone A D004
2000 Zone D D002
5000 Zone A 

### Test 2: Mock Record with Sort by Zone

In [46]:
records_reverse=np.empty(10, dtype=object)
records_reverse[0] = DeliveryRecord(500, "Zone J", "D021")
records_reverse[1] = DeliveryRecord(700, "Zone I", "D996")
records_reverse[2] = DeliveryRecord(900, "Zone H", "D001")
records_reverse[3] = DeliveryRecord(650, "Zone G", "D207")
records_reverse[4] = DeliveryRecord(300, "Zone F", "D406")
records_reverse[5] = DeliveryRecord(950, "Zone E", "D405")
records_reverse[6] = DeliveryRecord(100, "Zone D", "D504")
records_reverse[7] = DeliveryRecord(200, "Zone C", "D013")
records_reverse[8] = DeliveryRecord(300, "Zone B", "D044")
records_reverse[9] = DeliveryRecord(1000, "Zone A", "D001")

mockSortTest(records_reverse, attribute="address")


Before Quick Sorting:

500 Zone J D021
700 Zone I D996
900 Zone H D001
650 Zone G D207
300 Zone F D406
950 Zone E D405
100 Zone D D504
200 Zone C D013
300 Zone B D044
1000 Zone A D001

QUICKSORT Performance

Total Records  : 10
Execution Time: 0.000009 sec
Number of Comparisons: 21
Number of Swaps: 24


After Quick Sorting:

1000 Zone A D001
300 Zone B D044
200 Zone C D013
100 Zone D D504
950 Zone E D405
300 Zone F D406
650 Zone G D207
900 Zone H D001
700 Zone I D996
500 Zone J D021

Before Merge Sorting:

500 Zone J D021
700 Zone I D996
900 Zone H D001
650 Zone G D207
300 Zone F D406
950 Zone E D405
100 Zone D D504
200 Zone C D013
300 Zone B D044
1000 Zone A D001


MERGESORT Performance

Total Records  : 10
Execution Time: 0.000012 sec
Number of Comparisons: 15
Number of Swaps: 34

After Merge Sorting:

1000 Zone A D001
300 Zone B D044
200 Zone C D013
100 Zone D D504
950 Zone E D405
300 Zone F D406
650 Zone G D207
900 Zone H D001
700 Zone I D996
500 Zone J D021



### Test 3: 100 Parcels with Randomly Sorted Delivery Time (Quick and Merge)

In [47]:
sort1= Sorter()
data = sort1.makeArray('Samples/random_100.csv')
sort1.quickSort(data)
sort1.makeCSV('Samples/Output/result_parcels_100_random_quick.csv', data)

sort2= Sorter()
data = sort2.makeArray('Samples/random_100.csv')
sort2.mergeSort(data)
sort2.makeCSV('Samples/Output/result_parcels_100_random_merge.csv', data)


QUICKSORT Performance

Total Records  : 100
Execution Time: 0.000116 sec
Number of Comparisons: 653
Number of Swaps: 443


MERGESORT Performance

Total Records  : 100
Execution Time: 0.000172 sec
Number of Comparisons: 548
Number of Swaps: 672



### Test 4: 100 Parcels with Reversed Sorted Delivery Time (Quick and Merge)

In [48]:
sort1= Sorter()
data = sort1.makeArray('Samples/reversed_100.csv')
sort1.quickSort(data)
sort1.makeCSV('Samples/Output/result_parcels_100_reverse_quick.csv', data)

sort2= Sorter()
data = sort2.makeArray('Samples/reversed_100.csv')
sort2.mergeSort(data)
sort2.makeCSV('Samples/Output/result_parcels_100_reverse_merge.csv', data)



QUICKSORT Performance

Total Records  : 100
Execution Time: 0.000085 sec
Number of Comparisons: 514
Number of Swaps: 399


MERGESORT Performance

Total Records  : 100
Execution Time: 0.000141 sec
Number of Comparisons: 316
Number of Swaps: 672



### Test 5: 100 Parcels with Nearly Sorted Delivery Time (Quick and Merge)

In [49]:
sort1= Sorter()
data = sort1.makeArray('Samples/nearly_sorted_100.csv')
sort1.quickSort(data)
sort1.makeCSV('Samples/Output/result_parcels_100_nearly_sorted_quick.csv', data)

sort2= Sorter()
data = sort2.makeArray('Samples/nearly_sorted_100.csv')
sort2.mergeSort(data)
sort2.makeCSV('Samples/Output/result_parcels_100_nearly_sorted_merge.csv', data)


QUICKSORT Performance

Total Records  : 100
Execution Time: 0.000100 sec
Number of Comparisons: 851
Number of Swaps: 307


MERGESORT Performance

Total Records  : 100
Execution Time: 0.000153 sec
Number of Comparisons: 409
Number of Swaps: 672



### Test 6: 500 Parcels with Randomly Sorted Delivery Time (Quick and Merge)

In [50]:
sort1= Sorter()
data = sort1.makeArray('Samples/random_500.csv')
sort1.quickSort(data)
sort1.makeCSV('Samples/Output/result_parcels_500_random_quick.csv', data)

sort2= Sorter()
data = sort2.makeArray('Samples/random_500.csv')
sort2.mergeSort(data)
sort2.makeCSV('Samples/Output/result_parcels_500_random_merge.csv', data)


QUICKSORT Performance

Total Records  : 500
Execution Time: 0.000602 sec
Number of Comparisons: 4640
Number of Swaps: 2868


MERGESORT Performance

Total Records  : 500
Execution Time: 0.001152 sec
Number of Comparisons: 3871
Number of Swaps: 4488



### Test 7: 500 Parcels with Reversed Sorted Delivery Time (Quick and Merge)

In [51]:
sort1= Sorter()
data = sort1.makeArray('Samples/reversed_500.csv')
sort1.quickSort(data)
sort1.makeCSV('Samples/Output/result_parcels_500_reverse_quick.csv', data)

sort2= Sorter()
data = sort2.makeArray('Samples/reversed_500.csv')
sort2.mergeSort(data)
sort2.makeCSV('Samples/Output/result_parcels_500_reverse_merge.csv', data)



QUICKSORT Performance

Total Records  : 500
Execution Time: 0.000607 sec
Number of Comparisons: 4312
Number of Swaps: 2908


MERGESORT Performance

Total Records  : 500
Execution Time: 0.000887 sec
Number of Comparisons: 2625
Number of Swaps: 4488



### Test 8: 500 Parcels with Nearly Sorted Delivery Time (Quick and Merge)

In [52]:
sort1= Sorter()
data = sort1.makeArray('Samples/nearly_sorted_500.csv')
sort1.quickSort(data)
sort1.makeCSV('Samples/Output/result_parcels_500_nearly_sorted_quick.csv', data)

sort2= Sorter()
data = sort2.makeArray('Samples/reversed_500.csv')
sort2.mergeSort(data)
sort2.makeCSV('Samples/Output/result_parcels_500_nearly_sorted_merge.csv', data)


QUICKSORT Performance

Total Records  : 499
Execution Time: 0.000710 sec
Number of Comparisons: 6291
Number of Swaps: 2494


MERGESORT Performance

Total Records  : 500
Execution Time: 0.000989 sec
Number of Comparisons: 2625
Number of Swaps: 4488



### Test 9: 1000 Parcels with Randomly Sorted Delivery Time (Quick and Merge)

In [53]:
sort1= Sorter()
data = sort1.makeArray('Samples/random_1000.csv')
sort1.quickSort(data)
sort1.makeCSV('Samples/Output/result_parcels_1000_random_quick.csv', data)

sort2= Sorter()
data = sort2.makeArray('Samples/random_1000.csv')
sort2.mergeSort(data)
sort2.makeCSV('Samples/Output/result_parcels_1000_random_merge.csv', data)


QUICKSORT Performance

Total Records  : 1000
Execution Time: 0.001483 sec
Number of Comparisons: 11251
Number of Swaps: 6366


MERGESORT Performance

Total Records  : 1000
Execution Time: 0.002283 sec
Number of Comparisons: 8723
Number of Swaps: 9976



### Test 10: 1000 Parcels with Reversed Sorted Delivery Time (Quick and Merge)

In [54]:
sort1= Sorter()
data = sort1.makeArray('Samples/reversed_1000.csv')
sort1.quickSort(data)
sort1.makeCSV('Samples/Output/result_parcels_1000_reverse_quick.csv', data)

sort2= Sorter()
data = sort2.makeArray('Samples/reversed_1000.csv')
sort2.mergeSort(data)
sort2.makeCSV('Samples/Output/result_parcels_1000_reverse_merge.csv', data)



QUICKSORT Performance

Total Records  : 1000
Execution Time: 0.001349 sec
Number of Comparisons: 10069
Number of Swaps: 6501


MERGESORT Performance

Total Records  : 1000
Execution Time: 0.001877 sec
Number of Comparisons: 5765
Number of Swaps: 9976



### Test 11: 1000 Parcels with Nearly Sorted Delivery Time (Quick and Merge)

In [55]:
sort1= Sorter()
data = sort1.makeArray('Samples/nearly_sorted_1000.csv')
sort1.quickSort(data)
sort1.makeCSV('Samples/Output/result_parcels_1000_nearly_sorted_quick.csv', data)

sort2= Sorter()
data = sort2.makeArray('Samples/nearly_sorted_1000.csv')
sort2.mergeSort(data)
sort2.makeCSV('Samples/Output/result_parcels_1000_nearly_sorted_merge.csv', data)


QUICKSORT Performance

Total Records  : 1000
Execution Time: 0.001708 sec
Number of Comparisons: 17361
Number of Swaps: 5067


MERGESORT Performance

Total Records  : 1000
Execution Time: 0.002709 sec
Number of Comparisons: 5676
Number of Swaps: 9976

